# Member 4: Model Training & 5-Fold Validation Pipeline
**Project 7**: SBERT Sentence Embeddings + Cross-Project Setting + Random Split

### Member 4 Responsibilities (Strict Protocol):
1. **Development Data ONLY**: Test sets are **strictly isolated** and never touched.
2. **Simple Baseline**: Median dummy predictor benchmark on development data.
3. **3-Layer 1D CNN**: Architecture specified in Project Guidelines Section 3.1.
4. **5-Fold Cross-Validation**: 4 folds for training, 1 fold for validation across development data.
5. **Model Serialization**: Train the final model on full development data and save checkpoints to `models/` for Member 5 to load without retraining.

In [1]:
import os
import glob
import json
import shutil
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, median_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')

# Automatically locate uploaded dataset files
emb_files = glob.glob('/kaggle/input/**/all_embeddings.npy', recursive=True) + glob.glob('./**/all_embeddings.npy', recursive=True)
if not emb_files:
    raise FileNotFoundError('Could not find all_embeddings.npy! Please attach kaggle_dataset.zip to this notebook.')

data_dir = os.path.dirname(emb_files[0])
print('Data Directory:', data_dir)

all_embeddings = np.load(os.path.join(data_dir, 'all_embeddings.npy'))
issues_df = pd.read_csv(os.path.join(data_dir, 'issues_clean.csv'))

print(f'Total issues in database: {len(issues_df):,} across {issues_df["project_id"].nunique()} projects.')
print(f'Feature matrix shape: {all_embeddings.shape}')

Using Device: cuda
GPU Name: Tesla T4
Data Directory: /kaggle/input/datasets/musa62/sbert-story-points
Total issues in database: 63,011 across 39 projects.
Feature matrix shape: (63011, 384)


## 1. Model Architecture: 3-Layer 1D CNN (Section 3.1)

In [2]:
class StoryPointCNN(nn.Module):
    """
    3-Layer 1D CNN regressor for SBERT sentence embeddings (384 dimensions).
    Conforms directly to Project Guidelines Section 3.1.
    """
    def __init__(self, input_dim=384, dropout=0.2):
        super(StoryPointCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool1d(2)  # 384 -> 192

        self.conv2 = nn.Conv1d(64, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool1d(2)  # 192 -> 96

        self.conv3 = nn.Conv1d(32, 16, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool1d(2)  # 96 -> 48

        self.fc = nn.Sequential(
            nn.Linear(16 * (input_dim // 8), 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x).squeeze(-1)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    return total_loss / len(loader.dataset)

def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_b, in loader:
            preds.extend(model(X_b.to(device)).cpu().numpy())
    return np.array(preds)

## 2. Member 4 Core Workflow: 5-Fold Cross-Validation on Development Data Only
For each project pair:
* We isolate the development data (all issues except the held-out project).
* We run **5-Fold Cross-Validation** exclusively on development data.
* We fit the final model on 100% of the development data.
* We save the model weights and JSON configuration to `models/` for Member 5.

In [3]:
MODELS_DIR = '/kaggle/working/models'
os.makedirs(MODELS_DIR, exist_ok=True)

projects = sorted(issues_df['project_id'].unique())
print(f'Total projects to process: {len(projects)}')

cv_summary_records = []
EPOCHS = 5
BATCH_SIZE = 512
LR = 1e-3

for idx, target_proj in enumerate(projects, start=1):
    pair_name = f'pair_{idx:02d}'
    pair_dir = os.path.join(MODELS_DIR, pair_name)
    os.makedirs(pair_dir, exist_ok=True)

    # STRICT RULE: Load ONLY development data (held-out project is NEVER touched)
    dev_mask = (issues_df['project_id'] != target_proj).values
    X_dev = all_embeddings[dev_mask]
    y_dev = issues_df.loc[dev_mask, 'story_points'].values.astype(np.float32)

    # 1. Simple Baseline (Median Predictor on Development Data)
    dev_median = float(np.median(y_dev))
    baseline_preds = np.full_like(y_dev, fill_value=dev_median)
    baseline_mae = float(mean_absolute_error(y_dev, baseline_preds))
    baseline_mdae = float(median_absolute_error(y_dev, baseline_preds))

    # 2. 5-Fold Cross-Validation on Development Data
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_dev), start=1):
        train_loader = DataLoader(
            TensorDataset(torch.tensor(X_dev[train_idx], dtype=torch.float32).unsqueeze(1),
                          torch.tensor(y_dev[train_idx], dtype=torch.float32)),
            batch_size=BATCH_SIZE,
            shuffle=True
        )
        val_loader = DataLoader(
            TensorDataset(torch.tensor(X_dev[val_idx], dtype=torch.float32).unsqueeze(1)),
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        cv_model = StoryPointCNN(input_dim=384).to(device)
        optimizer = torch.optim.Adam(cv_model.parameters(), lr=LR, weight_decay=1e-4)
        criterion = nn.L1Loss()

        for epoch in range(EPOCHS):
            train_epoch(cv_model, train_loader, optimizer, criterion, device)

        val_preds = predict(cv_model, val_loader, device)
        val_mae = float(mean_absolute_error(y_dev[val_idx], val_preds))
        val_mdae = float(median_absolute_error(y_dev[val_idx], val_preds))
        fold_metrics.append({'fold': fold, 'mae': round(val_mae, 4), 'mdae': round(val_mdae, 4)})

    cv_mean_mae = float(np.mean([m['mae'] for m in fold_metrics]))
    cv_std_mae = float(np.std([m['mae'] for m in fold_metrics]))
    cv_mean_mdae = float(np.mean([m['mdae'] for m in fold_metrics]))
    cv_std_mdae = float(np.std([m['mdae'] for m in fold_metrics]))

    # 3. Fit Final Selected Model on 100% Development Data
    full_train_loader = DataLoader(
        TensorDataset(torch.tensor(X_dev, dtype=torch.float32).unsqueeze(1),
                      torch.tensor(y_dev, dtype=torch.float32)),
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    final_model = StoryPointCNN(input_dim=384).to(device)
    optimizer = torch.optim.Adam(final_model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.L1Loss()

    for epoch in range(EPOCHS):
        train_epoch(final_model, full_train_loader, optimizer, criterion, device)

    # 4. Save Final Model & Configuration for Member 5
    model_save_path = os.path.join(pair_dir, 'story_point_cnn.pt')
    torch.save(final_model.state_dict(), model_save_path)

    config_payload = {
        'project': 'Project 7 (SBERT Sentence Embeddings + Cross-Project + Random Split)',
        'pair': pair_name,
        'held_out_project': target_proj,
        'development_records': len(y_dev),
        'model_architecture': 'StoryPointCNN (3-Layer 1D CNN)',
        'input_dimension': 384,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LR,
        'random_seed': 42,
        'baseline_median': {
            'constant_value': dev_median,
            'mae': round(baseline_mae, 4),
            'mdae': round(baseline_mdae, 4)
        },
        'cv_validation_results': {
            'folds': fold_metrics,
            'mean_mae': round(cv_mean_mae, 4),
            'std_mae': round(cv_std_mae, 4),
            'mean_mdae': round(cv_mean_mdae, 4),
            'std_mdae': round(cv_std_mdae, 4)
        },
        'saved_model_file': 'story_point_cnn.pt'
    }

    with open(os.path.join(pair_dir, 'model_config.json'), 'w') as f:
        json.dump(config_payload, f, indent=4)

    cv_summary_records.append({
        'pair': pair_name,
        'held_out_project': target_proj,
        'dev_records': len(y_dev),
        'baseline_mae': round(baseline_mae, 4),
        'baseline_mdae': round(baseline_mdae, 4),
        'cv_mean_mae': round(cv_mean_mae, 4),
        'cv_std_mae': round(cv_std_mae, 4),
        'cv_mean_mdae': round(cv_mean_mdae, 4),
        'cv_std_mdae': round(cv_std_mdae, 4)
    })

    print(f'[{idx:02d}/{len(projects)}] {pair_name} (Target: {target_proj:<12}) | Base MdAE: {baseline_mdae:.2f} -> CV MdAE: {cv_mean_mdae:.2f} (+/- {cv_std_mdae:.2f})')

summary_df = pd.DataFrame(cv_summary_records)
summary_df.to_csv(os.path.join(MODELS_DIR, 'cv_summary_39_pairs.csv'), index=False)
print('\n' + '=' * 70)
print('MEMBER 4 TRAINING & 5-FOLD CV COMPLETE FOR ALL 39 PAIRS!')
print('=' * 70)

Total projects to process: 39
[01/39] pair_01 (Target: ALOY        ) | Base MdAE: 2.00 -> CV MdAE: 1.80 (+/- 0.16)
[02/39] pair_02 (Target: APIKIT      ) | Base MdAE: 2.00 -> CV MdAE: 1.88 (+/- 0.06)
[03/39] pair_03 (Target: APSTUD      ) | Base MdAE: 2.00 -> CV MdAE: 1.79 (+/- 0.09)
[04/39] pair_04 (Target: BAM         ) | Base MdAE: 2.00 -> CV MdAE: 1.75 (+/- 0.08)
[05/39] pair_05 (Target: BE          ) | Base MdAE: 2.00 -> CV MdAE: 1.89 (+/- 0.04)
[06/39] pair_06 (Target: CLI         ) | Base MdAE: 2.00 -> CV MdAE: 1.80 (+/- 0.10)
[07/39] pair_07 (Target: CLOV        ) | Base MdAE: 2.00 -> CV MdAE: 1.88 (+/- 0.06)
[08/39] pair_08 (Target: COMPASS     ) | Base MdAE: 2.00 -> CV MdAE: 1.72 (+/- 0.17)
[09/39] pair_09 (Target: CONFCLOUD   ) | Base MdAE: 2.00 -> CV MdAE: 1.81 (+/- 0.14)
[10/39] pair_10 (Target: CONFSERVER  ) | Base MdAE: 2.00 -> CV MdAE: 1.78 (+/- 0.19)
[11/39] pair_11 (Target: CWD         ) | Base MdAE: 2.00 -> CV MdAE: 1.84 (+/- 0.07)
[12/39] pair_12 (Target: CXX       

## 3. Member 4 Validation Results Summary Table (Development Data Only)
This is the official table of validation results across all 39 development sets.

In [4]:
print('========================================================================')
print('MEMBER 4: 5-FOLD CROSS-VALIDATION SUMMARY (DEVELOPMENT DATA ONLY)')
print('========================================================================')
print(f'Overall Baseline Mean MdAE:  {summary_df["baseline_mdae"].mean():.4f} (+/- {summary_df["baseline_mdae"].std():.4f})')
print(f'Overall 3-Layer CNN Mean MdAE: {summary_df["cv_mean_mdae"].mean():.4f} (+/- {summary_df["cv_mean_mdae"].std():.4f})')
reduction = (1 - summary_df['cv_mean_mdae'].mean() / summary_df['baseline_mdae'].mean()) * 100
print(f'Average Error Reduction across Development Sets: {reduction:.2f}%')
print('========================================================================')
summary_df

MEMBER 4: 5-FOLD CROSS-VALIDATION SUMMARY (DEVELOPMENT DATA ONLY)
Overall Baseline Mean MdAE:  2.0000 (+/- 0.0000)
Overall 3-Layer CNN Mean MdAE: 1.7758 (+/- 0.0693)
Average Error Reduction across Development Sets: 11.21%


,pair,held_out_project,dev_records,baseline_mae,baseline_mdae,cv_mean_mae,cv_std_mae,cv_mean_mdae,cv_std_mdae
0,pair_01,ALOY,62708,429.9740,2.0,429.9744,851.6377,1.7996,0.1555
1,pair_02,APIKIT,62546,431.0821,2.0,431.1029,854.1010,1.8848,0.0614
2,pair_03,APSTUD,62172,433.6062,2.0,433.6004,859.0274,1.7870,0.0935
3,pair_04,BAM,62495,431.4352,2.0,431.4558,854.6619,1.7528,0.0848
4,pair_05,BE,62639,430.4490,2.0,430.4507,852.6636,1.8902,0.0391
5,pair_06,CLI,62654,430.3427,2.0,430.3533,852.5274,1.7976,0.1046
6,pair_07,CLOV,62629,430.4995,2.0,430.5024,852.8996,1.8784,0.0610
7,pair_08,COMPASS,62512,431.3209,2.0,431.3227,854.4590,1.7156,0.1714
8,pair_09,CONFCLOUD,62671,430.2289,2.0,430.2508,852.3838,1.8063,0.1425
9,pair_10,CONFSERVER,62366,432.3282,2.0,432.3525,856.5261,1.7770,0.1881


## 4. Package Models for Member 5
This creates `models_for_member_5.zip` containing all trained `.pt` models and `model_config.json` files so Member 5 can load them and run evaluation without retraining.

In [5]:
zip_output_path = '/kaggle/working/models_for_member_5'
shutil.make_archive(zip_output_path, 'zip', MODELS_DIR)
zip_file = zip_output_path + '.zip'
print(f'SUCCESS! Created handoff zip file for Member 5: {zip_file}')
print(f'Size: {os.path.getsize(zip_file) / (1024 * 1024):.2f} MB')
print('You can download this zip file directly from Kaggle and send it to Member 5.')

SUCCESS! Created handoff zip file for Member 5: /kaggle/working/models_for_member_5.zip
Size: 8.30 MB
You can download this zip file directly from Kaggle and send it to Member 5.
